# Readme Vorbereitung gocfl create Befehle

Dieses Jupyiter Notebook erstellt die gocfl create-Befehle für eine bestimmte Collection. 
Die Doku für den Aufbau eines gocfl create Befehls befindet sich hier: https://github.com/je4/gocfl/blob/main/docs/create.md

## config.py

In der Config wird die aktuell zu verarbeitende Collection sowie diverse Dateipfade konfiguriert. Bspw. für E-Rara:

    collection = 'e-rara'
    root = 'zhb_archiv/e-rara'
    sigpath = 'fulldump/signatures.txt'

## signature

Die Signature ist zentral für die Erstellung des storage roots, sowie das Auffinden der Objekt-Pfade, Metadaten und Info-Dateien. Grundsätzlich sollte für jede Collection eine Textdatei namens '/signatures.txt' mit den signatures vorliegen. Diese werden entweder durch ein anderes Jupyter Notebook konfiguriert oder können von Hand erstellt werden.
Der Dateipfad kann konfiguriert werden. 

## storage root

Hier wird davon ausgegangen, dass der storage root ein ZIP file sein soll. Für jede signature wird ein storage_root angelegt. Der Root-Path kann konfiguriert werden. 


###  Create Befehl für gocfl generieren

Die gocfl create Befehle für alle Signaturen werden gemäss https://github.com/je4/gocfl/blob/main/docs/create.md erstellt. Der Pfad für die Config.toml kann konfiguriert werden.  

Muster:

    gocfl create ./archiv.zip ./object metadata:./metadata --config ./config/gocfl.toml -i 'signature'  --ext-NNNN-metafile-source ./info.json


In [3]:
import config
import os
from zipfile import ZipFile
import shutil

#prepare archive structure

ingest = config.ingest
root = config.root
archive = config.archive
collection = config.collection
sig_path = config.sigpath
sig_file = f'{ingest}/{sig_path}/signatures.txt'
md_path =  config.mdpath
md_format = config.mdformat
info_path = config.infopath
obj_path = config.objpath
gocfl_config = config.configfile

if os.path.exists(archive):
    print(f"Existing ZHB Archive directory: {archive}")    

else:
    os.mkdir(archive)
    print(f"ZHB Archive created: {archive}")

collection_path =  os.path.join(archive, collection)

if os.path.exists(collection_path):
        print(f"Existing collection directory: {collection_path}")
else:
    os.mkdir(collection_path)
    print(f"Collection directory created: {collection_path}")

if os.path.exists('gocfl_create'):
    shutil.rmtree('gocfl_create')
os.mkdir('gocfl_create')
    
if os.path.exists('gocfl_update'):
    shutil.rmtree('gocfl_update')    
os.mkdir('gocfl_update')

if os.path.exists('gocfl_errors'):
    shutil.rmtree('gocfl_errors')    
os.mkdir('gocfl_errors')

# read line from signaturesfile
with open(sig_file, 'r') as file:
    
    print("\nOpening file:",sig_file, "\n")
    for row in file:
        signature = row.strip()
        print("Signature:",signature)
        
        # create filepaths for metadata, info.json, objects:
        metadata_file = f'{ingest}/{md_path}/{signature}/{signature}.{md_format}'
        print("           Metadata: ",metadata_file)
        info_file = f'{ingest}/{info_path}/{signature}.json'        
        print("           Info.json: ",info_file)
        object_path = f'{ingest}/{obj_path}/'
        print("           Object path: ",object_path)
        
        # find the corresponding file in objects:
        file_start = signature[4:]
        file_nr = 0
        for object_file in os.listdir(object_path):            
            if object_file.startswith(file_start):
                file_nr = file_nr + 1
                object_name = object_file
                print(f"           File_path {file_nr}: {object_name}")
                         
            # TODO: handle more than 1 file for a signature, file not found
            if file_nr > 1:
                print(f"*****************More than 1 file for {signature}")
                
        if file_nr == 0:
            print(f"************* File starting with {signature} not found!")
            with open(f'gocfl_errors/file_not_found_{signature}.txt', 'w') as file:
                file.write(signature)
                print(f"************ Signature added to 'gocfl_errors/file_not_found_{signature}.txt'")
            continue 
        
        # create storage root for this object
        storage_root = f'{archive}/{collection}/{signature}.zip'
            
        if os.path.exists(storage_root):
            # update procedure necessary, write signature to update file and continue
            print("*********** Storage root exists already, use gocfl update!")
            with open(f'gocfl_update/gocfl_update_{signature}.txt', 'w') as file:
                file.write(signature)
                print(f"*********** Signature added to 'update/ocfl_update_{signature}.txt'")
            continue    
            
        else:
            with ZipFile(storage_root, 'w') as zipfile:
                print("           Storage root created: ",storage_root)
        
        # create string
        if object_name:
            
            create_string = f'gocfl create {root}/{storage_root} {root}/{object_path}{object_name} metadata: {root}/{metadata_file} --config {root}/{gocfl_config} -i "{signature}"  --ext-NNNN-metafile-source {root}/{info_file}'
            print('\n###############\n',create_string, '\n###############\n')
            create_file = f'gocfl_create/gocfl_create_{signature}.txt'
            with open(create_file, 'w') as file:
                file.write(create_string)
        else:
            print("no create string possible")
                

ZHB Archive created: zhb_dlza
Collection directory created: zhb_dlza\sosa_e-rara

Opening file: e-rara/files/signatures.txt 

Signature: zhb_10_3931_e-rara-86409
           Metadata:  e-rara/metadata/zhb_10_3931_e-rara-86409/zhb_10_3931_e-rara-86409.xml
           Info.json:  e-rara/info/zhb_10_3931_e-rara-86409.json
           Object path:  e-rara/objects/
           File_path 1: 10_3931_e-rara-86409_20201027T114533_master_ver1.zip
           Storage root created:  zhb_dlza/sosa_e-rara/zhb_10_3931_e-rara-86409.zip

###############
 gocfl create P:/temp/zhb_dlza/sosa_e-rara/zhb_10_3931_e-rara-86409.zip P:/temp/e-rara/objects/10_3931_e-rara-86409_20201027T114533_master_ver1.zip metadata: P:/temp/e-rara/metadata/zhb_10_3931_e-rara-86409/zhb_10_3931_e-rara-86409.xml --config P:/temp/config/gocfl.toml -i "zhb_10_3931_e-rara-86409"  --ext-NNNN-metafile-source P:/temp/e-rara/info/zhb_10_3931_e-rara-86409.json 
###############

Signature: zhb_10_3931_e-rara-87374
           Metadata:  e-rara/